<a href="https://colab.research.google.com/github/AHMADCH011/flyrank-ml-internship/blob/main/work/notebooks%20/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHMADCH011/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Refresh / Content Opportunity Scoring (`w01`) — *which content items should be reviewed first for a refresh?*


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (`content_hash_id`), scored at a mid-month decision point.**
The raw table's grain is `report_date x client_hash_id x content_hash_id` — one measurement per
content item per day. My analysis row is a per-content-item summary I build by aggregating that
daily grain over a 15-day feature window; the daily grain itself is what I verify in section 3.

**Tables:** `fact_content_daily_performance` (the daily time series — this notebook reads only the
`month=2026-03` partition, a mid-panel month, per the leakage rule below), `dim_content` (content
metadata, joined on `content_hash_id`), `dim_clients` (per-client tracking-start dates and
`access_profile`, joined on `client_hash_id`).

**Time window:** decision point = **2026-03-15**. Feature window = **2026-03-01 to 2026-03-15**
(the only days my five features are allowed to read). Outcome window = **2026-03-16 to 2026-03-31**
(used only to build the teaching label in section 3 — never as a feature). I stay entirely inside
`month=2026-03` on purpose: the brief for this data explicitly warns that `_sample`
(`fact_content_daily_performance_sample`) is the panel's real **final month (June 2026)** — the
natural outcome window of any past-to-future label — so I never touch it while developing label
logic, and I'm treating it as a sealed test month for later weeks.

**What I'd predict or rank:** a proxy label, `declining_with_demand` — whether a content item's
impressions in the outcome window fall by more than 20% versus the feature window, among items
that had real demand to begin with (>=20 impressions in the feature window). The real lane output
is a **ranked queue** (which items to review first), not a single classification — this label is
a stand-in used here only to demonstrate verification and the leakage trap.

**One thing I deliberately exclude:** FlyRank's own product decision fields (`health_score`,
`priority_score`, `action_type`, refresh flags). They aren't shipped in this release at all, but
I'm naming the exclusion anyway because it's the one that matters most: feeding a product's own
decision into a model just teaches it to copy that decision — a circular result, not a discovery.


In [ ]:
# Connect to the warehouse and confirm the raw table sizes before trusting anything above.
%pip -q install duckdb

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    # mid-panel month ONLY -- never the _sample table for label logic (see section 1)
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:12} {n:>12,} rows')


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (feature window only) | Feature | Observed search signals, logged before the decision point |
| `ga4_data_available` (feature window only) | Feature | A fact about tracking infrastructure, not about the outcome — known before the decision point |
| `gsc_impressions` (outcome window) | Label input | Only used to compute `declining_with_demand`; never joined into the feature frame |
| `content_hash_id`, `client_hash_id` | Context | Join/group keys only — pseudonyms carry no signal themselves |
| `report_date` | Context | Used to build the feature/outcome window split, not as a model input |
| `access_profile`, `gsc_data_start`, `ga4_data_start` (`dim_clients`) | Context | Used to check availability and exclude ambiguous rows, not fed to a model |
| `health_score`, `priority_score`, `action_type`, refresh flags | Excluded | Not shipped in this release — and if they ever were, they're FlyRank's own product decision, not an observed outcome; using them would let a model copy the existing rule instead of finding its own signal |
| Anything from `fact_content_query_90d` | Excluded (this notebook) | Its 90-day window overlaps the snapshot's final months; aligning it correctly against my March window is next week's job, not this one |


In [ ]:
# Ground the field table above in the real columns, not memory.
print('dim_content columns:')
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()[['column_name','column_type']])
print()
print('fact_march columns:')
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']}").df()[['column_name','column_type']])


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims from section 1, three queries, on `month=2026-03`:

1. **Grain** — one row really is one `report_date x client_hash_id x content_hash_id`.
2. **Row count + date span** — how big this slice is and what dates it actually covers.
3. **Availability** — filtering GA4 rows with `IS TRUE` (not `= TRUE`, because the flag can be
   NULL, not just FALSE), and how many rows survive.


In [ ]:
# --- Query 1: grain probe -----------------------------------------------
# Zero rows back means the stated grain holds.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f'duplicate (date, client, content) combinations found: {len(dupes)}')
dupes


In [ ]:
# --- Query 2: row count + date span --------------------------------------
span = con.sql(f"""
    SELECT COUNT(*)                        AS rows_in_slice,
           COUNT(DISTINCT content_hash_id)  AS distinct_content_items,
           COUNT(DISTINCT client_hash_id)   AS distinct_clients,
           MIN(report_date)                AS min_date,
           MAX(report_date)                AS max_date
    FROM {TABLES['fact_march']}
""").df()
span


In [ ]:
# --- Query 3: availability, filtered with IS TRUE ------------------------
# ga4_data_available can be TRUE, FALSE, or NULL (client never GA4-tracked in this window) --
# '= TRUE' silently drops NULLs into the wrong bucket. IS TRUE handles it correctly.
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS ga4_available_rows,
           SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_rows,
           SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS ga4_null_rows
    FROM {TABLES['fact_march']}
""").df()
avail['ga4_available_share'] = avail['ga4_available_rows'] / avail['total_rows']
avail


### Five features

Built from the **feature window only** (2026-03-01 to 2026-03-15), per content item, with a
minimum-volume floor (>=20 feature-window impressions) so low-volume noise doesn't masquerade as
signal — per the lane guide's rule that every tier read needs a stated volume floor.


In [ ]:
feature_frame = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions)                                              AS impressions_first15,
           SUM(gsc_clicks)                                                   AS clicks_first15,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)     AS avg_position_first15,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_first15,
           AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END)     AS ga4_available_share_first15
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING impressions_first15 >= 20
""").df()
print(f'{len(feature_frame):,} content items with enough feature-window demand')
feature_frame.head()


**Available when?** — one line per feature:

1. `impressions_first15` — knowable at the decision moment because it only sums `gsc_impressions`
   already logged on or before 2026-03-15.
2. `clicks_first15` — same reasoning: a sum over dates strictly inside the feature window.
3. `avg_position_first15` — knowable at decision time because it averages only positions recorded
   in the feature window (rows where `gsc_avg_position = 0`, i.e. "no position data," are excluded
   from the average, not treated as rank zero).
4. `days_with_impressions_first15` — a count of past days with visibility; every day counted has
   already happened by the decision point.
5. `ga4_available_share_first15` — knowable at decision time because it reflects whether GA4
   tracking was active for this item's client during the feature window — a fact about data
   collection infrastructure, established before 2026-03-15, not about what happens after it.


### The trap: one label-derived column, on purpose

First build the teaching label — `declining_with_demand` — strictly from the **outcome window**
(2026-03-16 to 2026-03-31), joined onto the honest feature frame above. Then score the five clean
features. Then add ONE column that is computed from the label's own inputs, watch the score jump
toward perfect, and delete it.


In [ ]:
# Build the label from the OUTCOME window only.
outcome = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_last15
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
""").df()

labeled = feature_frame.merge(outcome, on=['content_hash_id', 'client_hash_id'], how='left')
labeled['impressions_last15'] = labeled['impressions_last15'].fillna(0)
labeled['declining_with_demand'] = (
    labeled['impressions_last15'] < 0.8 * labeled['impressions_first15']
).astype(int)

print('base rate (predict majority class every time):',
      round(max(labeled['declining_with_demand'].mean(), 1 - labeled['declining_with_demand'].mean()), 3))
labeled['declining_with_demand'].value_counts()


In [ ]:
# Honest score: the five clean features only.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

HONEST_FEATURES = ['impressions_first15', 'clicks_first15', 'avg_position_first15',
                    'days_with_impressions_first15', 'ga4_available_share_first15']

model_data = labeled.dropna(subset=HONEST_FEATURES).copy()
X = model_data[HONEST_FEATURES]
y = model_data['declining_with_demand']

honest_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring='roc_auc')
print(f'HONEST features, 5-fold CV ROC AUC: {honest_scores.mean():.3f}')


In [ ]:
# Now spring the trap: add ONE column derived from the label's own inputs.
# impressions_last15 is literally the number the label was thresholded from -- it should never
# be a feature, since it isn't knowable until AFTER the decision point.
LEAKY_FEATURES = HONEST_FEATURES + ['impressions_last15']

X_leak = model_data[LEAKY_FEATURES]
leaky_scores = cross_val_score(LogisticRegression(max_iter=1000), X_leak, y, cv=5, scoring='roc_auc')
print(f'HONEST features only,         ROC AUC: {honest_scores.mean():.3f}')
print(f'+ impressions_last15 (LEAK),  ROC AUC: {leaky_scores.mean():.3f}')


In [ ]:
# Delete the leaked column and keep the honest number. This is the number that goes forward.
del X_leak, leaky_scores, LEAKY_FEATURES

print('Leak-free feature set kept for future weeks:', HONEST_FEATURES)
print(f'Honest ROC AUC to beat: {honest_scores.mean():.3f} (base rate: '
      f'{round(max(y.mean(), 1 - y.mean()), 3)})')


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: `ga4_available_share_first15` conflates two different situations into one
number.** A content item can show 0% GA4 availability because its client genuinely has no GA4
traffic that half-month, *or* because that client's `ga4_data_start` simply hadn't arrived yet by
2026-03-15 — i.e. "not tracked yet" and "tracked and idle" both collapse to the same feature
value. Query 3 above shows this isn't a corner case: a real share of March rows carry
`ga4_data_available = NULL` (not FALSE), meaning GA4 status itself is unknown for that row, not
just absent. Telling these apart would need joining `dim_clients.ga4_data_start` against each
row's `report_date` — worth doing before this feature is trusted in a real model, but out of
scope for this contract notebook. More generally, this is one 15/16-day split of one mid-panel
month: it cannot separate a real decline from ordinary within-month noise or seasonality (per the
lane guide, section 7) — that needs a longer, forward-looking window, which is next week's work.


In [ ]:
# Quick evidence for the limitation above: how much of March GA4 status is genuinely NULL,
# and how unevenly clients' GA4 tracking-start dates are spread.
null_share = con.sql(f"""
    SELECT ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END)
                 / COUNT(*), 2) AS pct_ga4_status_null
    FROM {TABLES['fact_march']}
""").df()
null_share


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
